In [1]:
# ============================================================
# NLP + K-MEANS CLUSTERING
# Dataset: inconsistent-column-number.csv
# ============================================================

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import csv
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")

print("Libraries imported successfully!")


# ============================================================
# 2. LOAD CSV WITH INCONSISTENT NUMBER OF COLUMNS
# ============================================================

file_path = "inconsistent-column-number.csv"

print("\nLoading dataset...")

with open(
    file_path,
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as file:

    reader = csv.reader(file)
    rows = list(reader)

if len(rows) == 0:
    raise ValueError("The CSV file is empty.")

print("Total rows including header:", len(rows))


# ============================================================
# 3. FIND MAXIMUM NUMBER OF COLUMNS
# ============================================================

column_counts = [len(row) for row in rows]

max_columns = max(column_counts)

print("Maximum number of columns:", max_columns)

print("\nColumn count distribution:")

count_distribution = pd.Series(column_counts).value_counts().sort_index()

print(count_distribution)


# ============================================================
# 4. CREATE HEADER
# ============================================================

header = rows[0]

original_header_length = len(header)

print("\nOriginal header:")
print(header)

# If the header has fewer columns than some data rows,
# create names for the additional columns.

if len(header) < max_columns:

    for i in range(len(header), max_columns):
        header.append(f"extra_column_{i+1}")


# If the header somehow has more columns, truncate it.

header = header[:max_columns]

print("\nFinal header:")
print(header)


# ============================================================
# 5. MAKE ALL ROWS THE SAME LENGTH
# ============================================================

fixed_rows = []

for row in rows[1:]:

    # If row has fewer columns, add missing values
    if len(row) < max_columns:

        row = row + [np.nan] * (max_columns - len(row))

    # If row has more columns, truncate it
    elif len(row) > max_columns:

        row = row[:max_columns]

    fixed_rows.append(row)


# ============================================================
# 6. CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(
    fixed_rows,
    columns=header
)

print("\nDataset loaded successfully!")
print("Dataset shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())


# ============================================================
# 7. DATASET INFORMATION
# ============================================================

print("\n" + "=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("\nShape:")
print(df.shape)

print("\nColumns:")
for column in df.columns:
    print("-", column)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())


# ============================================================
# 8. FIND THE MOST SUITABLE TEXT COLUMN
# ============================================================

print("\n" + "=" * 60)
print("FINDING TEXT COLUMN")
print("=" * 60)

text_scores = {}

for column in df.columns:

    values = df[column].dropna().astype(str)

    if len(values) == 0:
        text_scores[column] = 0
        continue

    # Average length of values
    average_length = values.str.len().mean()

    # Number of unique values
    unique_ratio = values.nunique() / len(values)

    # Score based primarily on text length
    text_scores[column] = average_length


print("\nAverage text length by column:")

for column, score in text_scores.items():
    print(f"{column}: {score:.2f}")


# Select column with the highest average text length

text_column = max(
    text_scores,
    key=text_scores.get
)

print("\nAutomatically selected text column:")
print(text_column)


# ============================================================
# OPTIONAL:
# If you know the exact text column, replace the previous
# selection with:
#
# text_column = "your_column_name"
# ============================================================


# ============================================================
# 9. DISPLAY SELECTED TEXT COLUMN
# ============================================================

print("\nSample text data:")

display(
    df[[text_column]].head(10)
)


# ============================================================
# 10. TEXT CLEANING FUNCTION
# ============================================================

def clean_text(text):

    # Convert to string
    text = str(text)

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+|https\S+",
        " ",
        text
    )

    # Remove email addresses
    text = re.sub(
        r"\S+@\S+",
        " ",
        text
    )

    # Remove HTML tags
    text = re.sub(
        r"<.*?>",
        " ",
        text
    )

    # Remove numbers
    text = re.sub(
        r"\d+",
        " ",
        text
    )

    # Keep only letters and spaces
    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    # Remove leading/trailing spaces
    text = text.strip()

    return text


# ============================================================
# 11. APPLY TEXT CLEANING
# ============================================================

print("\nCleaning text...")

df["clean_text"] = (
    df[text_column]
    .fillna("")
    .apply(clean_text)
)

print("Text cleaning completed!")

display(
    df[[text_column, "clean_text"]].head(10)
)


# ============================================================
# 12. REMOVE EMPTY TEXT ROWS
# ============================================================

before_rows = len(df)

df = df[
    df["clean_text"].str.strip() != ""
].copy()

df.reset_index(drop=True, inplace=True)

after_rows = len(df)

print("\nRows before removing empty text:", before_rows)
print("Rows after removing empty text:", after_rows)
print("Rows removed:", before_rows - after_rows)


if len(df) < 2:
    raise ValueError(
        "Not enough text rows for K-Means clustering."
    )


# ============================================================
# 13. TF-IDF VECTORIZATION
# ============================================================

print("\n" + "=" * 60)
print("TF-IDF NLP")
print("=" * 60)

print("\nConverting text into TF-IDF features...")

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    min_df=1,
    max_df=0.95,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(
    df["clean_text"]
)

print("TF-IDF completed!")

print("\nTF-IDF matrix shape:")
print(X.shape)

print(
    "\nNumber of vocabulary terms:",
    len(vectorizer.get_feature_names_out())
)


# ============================================================
# 14. SHOW TF-IDF FEATURES
# ============================================================

feature_names = vectorizer.get_feature_names_out()

print("\nFirst 50 TF-IDF features:")

print(
    feature_names[:50]
)


# ============================================================
# 15. DETERMINE MAXIMUM POSSIBLE K
# ============================================================

number_of_documents = X.shape[0]

# We need at least 2 clusters.
# Do not test more clusters than the number of documents.

max_k = min(
    10,
    number_of_documents - 1
)

if max_k < 2:
    raise ValueError(
        "The dataset must contain at least 3 non-empty text rows."
    )


# ============================================================
# 16. ELBOW METHOD
# ============================================================

print("\n" + "=" * 60)
print("ELBOW METHOD")
print("=" * 60)

k_values = range(2, max_k + 1)

inertias = []

for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    model.fit(X)

    inertias.append(
        model.inertia_
    )


# Plot elbow curve

plt.figure(figsize=(10, 6))

plt.plot(
    list(k_values),
    inertias,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("K-Means Elbow Method")

plt.xticks(
    list(k_values)
)

plt.grid(True)

plt.show()


# ============================================================
# 17. SILHOUETTE SCORE
# ============================================================

print("\n" + "=" * 60)
print("SILHOUETTE SCORE")
print("=" * 60)

silhouette_scores = []

for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(X)

    score = silhouette_score(
        X,
        labels
    )

    silhouette_scores.append(score)

    print(
        f"K = {k} --> Silhouette Score = {score:.4f}"
    )


# ============================================================
# 18. PLOT SILHOUETTE SCORES
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    list(k_values),
    silhouette_scores,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")

plt.title(
    "Silhouette Score for Different K Values"
)

plt.xticks(
    list(k_values)
)

plt.grid(True)

plt.show()


# ============================================================
# 19. SELECT BEST K
# ============================================================

best_k_index = np.argmax(
    silhouette_scores
)

best_k = list(k_values)[
    best_k_index
]

print("\nBest K based on silhouette score:")
print(best_k)

print(
    "Best silhouette score:",
    round(
        silhouette_scores[best_k_index],
        4
    )
)


# ============================================================
# 20. K-MEANS CLUSTERING
# ============================================================

print("\n" + "=" * 60)
print("K-MEANS CLUSTERING")
print("=" * 60)

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(X)

df["cluster"] = cluster_labels

print("\nK-Means completed!")

print("\nNumber of documents in each cluster:")

print(
    df["cluster"]
    .value_counts()
    .sort_index()
)


# ============================================================
# 21. DISPLAY CLUSTERED DATA
# ============================================================

print("\n" + "=" * 60)
print("CLUSTERED DATA")
print("=" * 60)

display(
    df[
        [
            text_column,
            "clean_text",
            "cluster"
        ]
    ].sort_values(
        "cluster"
    ).head(50)
)


# ============================================================
# 22. TOP WORDS IN EACH CLUSTER
# ============================================================

print("\n" + "=" * 60)
print("TOP WORDS IN EACH CLUSTER")
print("=" * 60)

terms = vectorizer.get_feature_names_out()

cluster_centers = kmeans.cluster_centers_

for cluster_number in range(best_k):

    # Get indexes of highest values
    top_indices = cluster_centers[
        cluster_number
    ].argsort()[-20:][::-1]

    top_words = [
        terms[index]
        for index in top_indices
    ]

    print(
        f"\nCluster {cluster_number}"
    )

    print(
        "Top words:"
    )

    print(
        ", ".join(top_words)
    )


# ============================================================
# 23. CREATE CLUSTER SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("CLUSTER SUMMARY")
print("=" * 60)

cluster_summary = (
    df["cluster"]
    .value_counts()
    .sort_index()
    .reset_index()
)

cluster_summary.columns = [
    "Cluster",
    "Number_of_documents"
]

display(cluster_summary)


# ============================================================
# 24. VISUALIZE CLUSTERS USING SVD
# ============================================================

print("\nCreating 2D cluster visualization...")

# TruncatedSVD is more suitable than normal PCA for
# sparse TF-IDF matrices.

svd = TruncatedSVD(
    n_components=2,
    random_state=42
)

X_2d = svd.fit_transform(X)


# ============================================================
# 25. PLOT CLUSTERS
# ============================================================

plt.figure(figsize=(12, 8))

scatter = plt.scatter(
    X_2d[:, 0],
    X_2d[:, 1],
    c=df["cluster"],
    alpha=0.7
)

plt.xlabel(
    "SVD Component 1"
)

plt.ylabel(
    "SVD Component 2"
)

plt.title(
    "K-Means Clustering of NLP Text Data"
)

plt.colorbar(
    scatter,
    label="Cluster"
)

plt.grid(True)

plt.show()


# ============================================================
# 26. PLOT NUMBER OF DOCUMENTS PER CLUSTER
# ============================================================

cluster_counts = (
    df["cluster"]
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(10, 6))

plt.bar(
    cluster_counts.index.astype(str),
    cluster_counts.values
)

plt.xlabel(
    "Cluster"
)

plt.ylabel(
    "Number of Documents"
)

plt.title(
    "Number of Documents per Cluster"
)

plt.grid(
    axis="y"
)

plt.show()


# ============================================================
# 27. SHOW SAMPLE DOCUMENTS FROM EACH CLUSTER
# ============================================================

print("\n" + "=" * 60)
print("SAMPLE DOCUMENTS FROM EACH CLUSTER")
print("=" * 60)

for cluster_number in range(best_k):

    print(
        f"\n{'=' * 20} CLUSTER {cluster_number} {'=' * 20}"
    )

    cluster_data = df[
        df["cluster"] == cluster_number
    ]

    # Display up to 5 examples
    samples = cluster_data[
        [text_column]
    ].head(5)

    for index, row in samples.iterrows():

        print(
            f"\nDocument {index}:"
        )

        print(
            str(row[text_column])[:500]
        )


# ============================================================
# 28. CALCULATE FINAL SILHOUETTE SCORE
# ============================================================

final_score = silhouette_score(
    X,
    df["cluster"]
)

print("\n" + "=" * 60)
print("FINAL MODEL EVALUATION")
print("=" * 60)

print(
    "Number of clusters:",
    best_k
)

print(
    "Silhouette Score:",
    round(
        final_score,
        4
    )
)


# ============================================================
# 29. SAVE CLUSTERED DATASET
# ============================================================

output_file = (
    "inconsistent-column-number_clustered.csv"
)

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8"
)

print(
    "\nClustered dataset saved as:"
)

print(
    output_file
)


# ============================================================
# 30. SAVE CLUSTER SUMMARY
# ============================================================

summary_file = (
    "cluster_summary.csv"
)

cluster_summary.to_csv(
    summary_file,
    index=False
)

print(
    "Cluster summary saved as:"
)

print(
    summary_file
)


# ============================================================
# 31. FINAL RESULTS
# ============================================================

print("\n" + "=" * 60)
print("FINAL RESULTS")
print("=" * 60)

print(
    f"Original rows: {before_rows}"
)

print(
    f"Rows used for NLP: {len(df)}"
)

print(
    f"Text column: {text_column}"
)

print(
    f"TF-IDF features: {X.shape[1]}"
)

print(
    f"Best number of clusters: {best_k}"
)

print(
    f"Silhouette score: {final_score:.4f}"
)

print(
    "\nDocuments per cluster:"
)

print(
    df["cluster"]
    .value_counts()
    .sort_index()
)

print(
    "\nDone!"
)

Libraries imported successfully!

Loading dataset...
Total rows including header: 3
Maximum number of columns: 3

Column count distribution:
2    1
3    2
Name: count, dtype: int64

Original header:
['number', 'string', 'boolean']

Final header:
['number', 'string', 'boolean']

Dataset loaded successfully!
Dataset shape: (2, 3)

First 5 rows:


,number,string,boolean
0,1,one,true
1,2,two,NaN



DATASET INFORMATION

Shape:
(2, 3)

Columns:
- number
- string
- boolean

Data types:
number     str
string     str
boolean    str
dtype: object

Missing values:
number     0
string     0
boolean    1
dtype: int64

FINDING TEXT COLUMN

Average text length by column:
number: 1.00
string: 3.00
boolean: 4.00

Automatically selected text column:
boolean

Sample text data:


,boolean
0,true
1,NaN



Cleaning text...
Text cleaning completed!


,boolean,clean_text
0,true,true
1,NaN,



Rows before removing empty text: 2
Rows after removing empty text: 1
Rows removed: 1


ValueError: Not enough text rows for K-Means clustering.